# BindCraft Binder Design — One-Off Sampling Example

Design a single protein binder against a target with the BindCraft pipeline
(AlphaFold2 hallucination + ProteinMPNN refinement + PyRosetta filtering).

- Paper: [Pacesa et al., Nature 2025](https://www.nature.com/articles/s41586-025-09429-6)
- Upstream repo (pinned commit `7cd4ace`): https://github.com/martinpacesa/BindCraft/tree/7cd4ace1b7407adf66a50dfefa47de2270f5e4a9

This example uses tiny iteration counts and `max_trajectories=1` for a
~10-30 minute run on an H100. For production runs (100 binders, 24-72h),
leave `BindCraftConfig` at its upstream defaults and set
`number_of_final_designs=100` on the input.

In [ ]:
from proto_tools.tools.structure_design.bindcraft import (
    BindCraftConfig,
    BindCraftInput,
    run_bindcraft_design,
)

## Input API Reference

`BindCraftInput` specifies the target and the desired binder properties.

| Field | Type | Default | Description |
|---|---|---|---|
| `target_pdb` | `str` | *(required)* | Target structure (file path or PDB-format string). |
| `target_chain` | `str` | `"A"` | Chain ID(s) of the frozen target (comma-separated for multi-chain). |
| `target_hotspot_residues` | `str \| None` | `None` | Comma-separated 1-indexed target residue positions the binder must contact (e.g. `"1-10,56,78"`). `None` = unrestricted. |
| `binder_lengths` | `tuple[int, int]` | `(65, 150)` | `(min, max)` binder length range. |
| `binder_name` | `str` | `"binder"` | Project identifier used as a prefix in output filenames. |
| `number_of_final_designs` | `int` | `100` | Target accepted-design count. Set to `1` for one-off sampling. |

The designed binder is always emitted as chain `B` — upstream BindCraft hardcodes this and does not expose it as a setting.

## Config API Reference

`BindCraftConfig` exposes 51 of upstream's 61 `settings_advanced` fields as typed knobs plus a
`filter_overrides` dict for per-metric threshold tweaks. The other 10 upstream fields are
file-output / cleanup toggles (animations / loss-curve plots / MPNN FASTAs / trajectory
pickles / zip archives / intermediate-PDB cleanup) and the `sample_models` debug flag —
hardcoded in the dispatch payload because proto-tools never surfaces those scratch
artifacts. The full surface is documented in the auto-generated reference page (link
below); the table here lists the 10 knobs you'll actually tune.

| Field | Type | Default | Description |
|---|---|---|---|
| `design_algorithm` | `Literal["2stage", "3stage", "4stage", "greedy", "mcmc"]` | `"4stage"` | Hallucination algorithm. |
| `max_trajectories` | `int \| bool` | `False` | Cap on hallucination trajectories. `False` = unlimited. Set to a small int for one-off sampling. |
| `soft_iterations` | `int` | `75` | Stage-1 (soft) hallucination iteration count. |
| `temporary_iterations` | `int` | `45` | Stage-2 (temporary) hallucination iteration count. |
| `hard_iterations` | `int` | `5` | Stage-3 (hard) hallucination iteration count. |
| `greedy_iterations` | `int` | `15` | Stage-4 (greedy) hallucination iteration count. |
| `num_seqs` | `int` | `20` | Number of MPNN sequences to sample per trajectory. |
| `max_mpnn_sequences` | `int` | `2` | Max MPNN sequences validated (and considered for acceptance) per trajectory. |
| `num_recycles_validation` | `int` | `3` | AF2 recycles during validation. Increase for higher confidence at cost of runtime. |
| `filter_overrides` | `dict[str, Any]` | `{}` | Per-metric threshold overrides merged on top of `default_filters.json`. |

See the [BindCraft documentation](https://bio-pro.mintlify.app/tools/structure-design/bindcraft) for the full 51-field user-facing surface (loss weights, contact geometry, template masking, beta-protein optimisation, etc.).

## Output API Reference

### `BindCraftOutput`

| Field | Type | Description |
|---|---|---|
| `designs` | `list[BindCraftDesign]` | Accepted binder designs (length is at most `BindCraftInput.number_of_final_designs`). |
| `n_trajectories_run` | `int` | Total trajectories attempted before stopping (success or hitting `max_trajectories`). |
| `n_designs_accepted` | `int` | Designs that passed all filters (equals `len(designs)`). |

Plus the standard `BaseToolOutput` metadata fields: `tool_id`, `execution_time`, `success`, `errors`.

### `BindCraftDesign`

| Field | Type | Description |
|---|---|---|
| `design_name` | `str` | Unique design identifier emitted by upstream (e.g. `"binder_l60_s12345_mpnn3"`). |
| `binder_sequence` | `str` | Designed binder amino-acid sequence (1-letter codes). |
| `structure` | `Structure` | Relaxed target+binder complex; B-factors are pLDDT on the 0-100 PDB scale. |
| `metrics` | `BindCraftMetrics` | Per-design averaged metrics evaluated by the filter check. |
| `seed` | `int` | Random seed of the trajectory that produced this design. |
| `interface_aas` | `dict[str, int]` | Amino-acid composition at the binder-target interface. |
| `interface_residues` | `list[int]` | 1-indexed binder residue positions at the interface. |

### `BindCraftMetrics`

Mirrors the `Average_*` columns in upstream's `final_design_stats.csv` (the columns the filter check evaluates against), with snake_case names. Primary metric: `avg_iptm`.

| Metric | Type | Range | Unit |
|---|---|---|---|
| `avg_plddt` | `float` | [0.0, 1.0] | fraction |
| `avg_ptm` | `float` | [0.0, 1.0] | fraction |
| `avg_iptm` | `float` | [0.0, 1.0] | fraction |
| `avg_pae` | `float` | ≥0.0 | Å |
| `avg_ipae` | `float` | ≥0.0 | Å |
| `avg_iplddt` | `float` | [0.0, 1.0] | fraction |
| `avg_ss_plddt` | `float` | [0.0, 1.0] | fraction |
| `avg_binder_plddt` | `float` | [0.0, 1.0] | fraction |
| `avg_binder_ptm` | `float` | [0.0, 1.0] | fraction |
| `avg_binder_pae` | `float` | ≥0.0 | Å |
| `binder_energy_score` | `float` | unbounded | REU |
| `dG` | `float` | unbounded | REU |
| `dSASA` | `float` | ≥0.0 | Å² |
| `dG_per_dSASA` | `float` | unbounded | REU/Å² |
| `interface_sasa_pct` | `float` | [0.0, 100.0] | percent |
| `interface_hydrophobicity` | `float` | [0.0, 1.0] | fraction |
| `surface_hydrophobicity` | `float` | [0.0, 1.0] | fraction |
| `shape_complementarity` | `float` | [0.0, 1.0] | fraction |
| `packstat` | `float` | [0.0, 1.0] | fraction |
| `n_interface_hbonds` | `float` | ≥0.0 | count |
| `interface_hbonds_pct` | `float` | [0.0, 100.0] | percent |
| `n_interface_unsat_hbonds` | `float` | ≥0.0 | count |
| `interface_unsat_hbonds_pct` | `float` | [0.0, 100.0] | percent |
| `n_interface_residues` | `float` | ≥0.0 | count |
| `binder_helix_pct` | `float` | [0.0, 100.0] | percent |
| `binder_betasheet_pct` | `float` | [0.0, 100.0] | percent |
| `binder_loop_pct` | `float` | [0.0, 100.0] | percent |
| `interface_helix_pct` | `float` | [0.0, 100.0] | percent |
| `interface_betasheet_pct` | `float` | [0.0, 100.0] | percent |
| `interface_loop_pct` | `float` | [0.0, 100.0] | percent |
| `hotspot_rmsd` | `float` | ≥0.0 | Å |
| `target_rmsd` | `float` | ≥0.0 | Å |
| `binder_rmsd` | `float` | ≥0.0 | Å |
| `unrelaxed_clashes` | `float` | ≥0.0 | count |
| `relaxed_clashes` | `float` | ≥0.0 | count |

In [ ]:
inputs = BindCraftInput(
    target_pdb="path/to/your/target.pdb",  # replace with a real path
    target_chain="A",
    target_hotspot_residues="56",          # 1-indexed residue position(s) on the target
    binder_lengths=(60, 70),               # (min, max)
    binder_name="example",
    number_of_final_designs=1,             # one-off sampling
)

config = BindCraftConfig(
    max_trajectories=1,                    # cap at 1 trajectory
    soft_iterations=10,                    # tiny iteration budget
    temporary_iterations=5,
    hard_iterations=2,
    greedy_iterations=2,
    num_seqs=2,
    max_mpnn_sequences=1,
    seed=42,                               # reproducible
)

result = run_bindcraft_design(inputs, config)
print(f"Trajectories run: {result.n_trajectories_run}")
print(f"Designs accepted: {result.n_designs_accepted}")

In [ ]:
if result.designs:
    design = result.designs[0]
    print(f"Design: {design.design_name}")
    print(f"Sequence ({len(design.binder_sequence)} aa): {design.binder_sequence}")
    print(f"Seed: {design.seed}")
    print(f"avg pLDDT: {design.metrics['avg_plddt']:.3f}")
    print(f"avg iPTM:  {design.metrics['avg_iptm']:.3f}")
    print(f"dG:        {design.metrics['dG']:.2f} REU")
    print(f"Shape complementarity: {design.metrics['shape_complementarity']:.3f}")
else:
    print("No designs passed the filter check — try more trajectories or relaxed filters.")

## Tuning runtime

BindCraft is a stochastic, filter-driven pipeline — most trajectories fail the filter check, so the wall-clock time depends on both the number of accepted designs you want and how strict the filters are. Three useful operating points:

- **One-off sampling (~10-30 min on H100)**: this notebook — `number_of_final_designs=1`, `max_trajectories=1`, tiny iteration counts. Good for smoke-testing the pipeline end-to-end on a new target.
- **Quick scan (~2-4h)**: `number_of_final_designs=5`, `max_trajectories=50`, leave iteration counts at upstream defaults. Enough designs to see if the target is tractable at all.
- **Production run (~24-72h)**: `number_of_final_designs=100`, no `max_trajectories` cap, default iteration counts. Matches the upstream paper's per-target budget.

In [ ]:
strict_inputs = BindCraftInput(
    target_pdb="path/to/your/target.pdb",
    target_chain="A",
    target_hotspot_residues="56",
    binder_lengths=(60, 70),
    binder_name="strict",
    number_of_final_designs=1,
)
strict_config = BindCraftConfig(
    max_trajectories=1,
    soft_iterations=10, temporary_iterations=5,
    hard_iterations=2, greedy_iterations=2,
    num_seqs=2, max_mpnn_sequences=1,
    seed=42,
    filter_overrides={
        "Average_pLDDT": {"threshold": 0.85, "higher": True},
        "Average_i_pTM": {"threshold": 0.6, "higher": True},
    },
)
strict_result = run_bindcraft_design(strict_inputs, strict_config)

In [ ]:
if result.designs:
    result.export("my_binder", file_format="pdb")  # writes a directory of PDBs + stats.json